# Vergleich zwischen allen Modellen

# 1 Preprocessing

## 1.1 Imports

In [ ]:
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

## 1.2 Einlesen aller Daten

In [ ]:
# Arbeitsverzeichnis (FENES_P2X)
current_dir = os.getcwd()

# Hilfsfunktion zum Erkennen ungültiger Namen (z. B. Leerzeichen, Minus etc.)
def sanitize_name(name):
    return name.replace("-", "_").replace(" ", "_").replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")

# Gehe durch alle Technologieordner (z. B. Methanol, Ammoniak, Fischer_Tropsch)
for technologie in os.listdir(current_dir):
    tech_path = os.path.join(current_dir, technologie)

    if not os.path.isdir(tech_path):
        continue

    for folder in os.listdir(tech_path):
        folder_path = os.path.join(tech_path, folder)

        if not os.path.isdir(folder_path):
            continue

        if re.search(r"vergleich|sensitiv", folder.lower()):
            print(f"⚠️  Übersprungen: {folder}")
            continue

        csv_path = os.path.join(folder_path, "Ergebnisse.csv")
        if os.path.isfile(csv_path):
            try:
                df = pd.read_csv(csv_path, sep=';', decimal=',')

                # Variable direkt erstellen
                var_name = sanitize_name(folder)
                globals()[var_name] = df.set_index("Variable")["Wert"]

                print(f"✅ {var_name} geladen mit {len(df)} Zeilen.")
            except Exception as e:
                print(f"❌ Fehler bei {csv_path}: {e}")
        else:
            print(f"⚠️  Keine Ergebnisse.csv in {folder_path}")



## 1.3 Farbvorgaben

In [ ]:
col_n2 = col_asu = "#e5742e"                # Corporate orange              oder Gas orange: "#e4932c"
col_pv = "#e4cb3a"                          # PV Gelb
col_wka = "#128fcf"                         # WKA Blau
col_h2 = col_pem = "#14689b"                # Corporate Blau                oder Wasser blau: "#1b5aa6"
col_nh3 = col_hb = "#822181"                # Corporate Lila                oder Fern/Nahwärme Lila: "#8f2f89"
col_h2s = "#79CADD"                         #Corporate Hellblau Alternativ: Stromsektor Rot  #E02229             bzw Gasspeicher rot
col_bat = "#8CBF3E"                         # ElChem Speicher grün
col_buy = col_chem = "#1D9838"              # Biomasse Grün
col_etc = "#828282"                         # Corporate Grau

col_co2 = col_mea = col_dac = "#050505"               # Diesel grau
col_saf = col_ft = "#c796c4"                # Coorporate lila blass                
col_el= col_sell = "#E02229"                # Stromsektor Rot
col_q = col_hp = "#B01920"                  # Dunkelrot (Geothermie rot) 
col_qs = "#B96428"                          # Braunkohle braun
col_naph = "#DADADA"                        # Coorporate grau blass
col_bio = "#C796C4"                         # Coorporate Lila blass

col_ch3oh = col_ms = "#e5742e"              # Gas orange                

## 1.4 Plot Vorlage

In [ ]:
def plot_modelvergleich(parameter, farbe, xlim, ylim, save_path, xlabel="Wert", title="Modellvergleich", multiplikator=1):
    # Definiere die Technologie-Reihenfolge
    tech_order = ["Ammoniak", "Methanol", "FischerTropsch"]

    # Technologie zuordnen
    def get_tech(name):
        for tech in tech_order:
            if name.startswith(tech):
                return tech
        return "Sonstige"

    # Modelle mit gewünschtem Parameter sammeln
    modelle = {
        name: globals()[name]
        for name in globals()
        if isinstance(globals()[name], pd.Series)
        and parameter in globals()[name]
    }

    if not modelle:
        print(f"⚠️ Keine Modelle mit Parameter '{parameter}' gefunden.")
        return

    # Sortieren: erst nach tech_order, dann nach -Wert
    modelle_sortiert = sorted(
        modelle.items(),
        key=lambda item: (
            tech_order.index(get_tech(item[0])),
            -item[1][parameter]
        )
    )

    # Extrahieren
    categories = list(name for name, _ in modelle_sortiert)
    values = [model[parameter] * multiplikator for _, model in modelle_sortiert]
    # values = [model[parameter] for _, model in modelle_sortiert]
    colors = [farbe] * len(values)

    # Trennlinien berechnen
    lines_and_labels = []
    last_tech = get_tech(categories[0])
    for i, name in enumerate(categories[1:], 1):
        tech = get_tech(name)
        if tech != last_tech:
            lines_and_labels.append((i - 0.5, tech))
            last_tech = tech

    # Gruppenbereiche für Hintergrund
    group_ranges = []
    last_tech = get_tech(categories[0])
    start = 0
    for i, name in enumerate(categories[1:], 1):
        tech = get_tech(name)
        if tech != last_tech:
            group_ranges.append((start, i, last_tech))
            start = i
            last_tech = tech
    group_ranges.append((start, len(categories), last_tech))

    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    bars = ax.barh(categories, values, color=colors, edgecolor='black')

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.1,
                bar.get_y() + bar.get_height() / 2,
                f"{width:,.2f}",
                va='center', ha='left', fontsize=8, color='black')

    # Stil & Achsen
    ax.spines['top'].set_color('lightgray')
    ax.spines['top'].set_linewidth(0.75)
    ax.spines['top'].set_linestyle("--")

    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(1)

    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(1)

    ax.spines['right'].set_color('lightgray')
    ax.spines['right'].set_linewidth(0.75)
    ax.spines['right'].set_linestyle("--")

    ax.set_axisbelow(True)
    ax.grid(which='major', linestyle='--', linewidth=0.75, color='lightgray')
    ax.grid(which='minor', linestyle=':', linewidth=0.25, color='lightgray')

    # Trennungslinien + Beschriftung
    for y, label in lines_and_labels:
        ax.axhline(y, color='gray', linestyle='-', linewidth=1)

    for start, end, tech in group_ranges:
        ax.axhspan(start - 0.5, end - 0.5, facecolor='lightgray', alpha=0.1)
        ax.text(xlim[1] - 0.1, (start + end - 1) / 2,
                tech, va='center', ha='right', fontsize=9, color='gray')

    # Beschriftung
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    plt.xlabel(xlabel, fontsize=10)
    plt.title(title, fontsize=10)
    plt.xticks(fontsize=9)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# 2 Eingangsdaten

# 3 Ergebnisse

## 3.1 GUV

In [ ]:
plot_modelvergleich(
    parameter="calc_GuV",
    farbe=col_etc,
    xlim=(-20, 5),
    ylim=(-0.5, 36.5),
    save_path="Vergleich_Bilder/calc_GuV_bar.png",
    xlabel="Million Euro in M€",
    title="Profit of the Energy Systems",
    multiplikator=1/1000000
)

# 4 Levelized Cost

## 4.1 Levelized Cost of Electricity

In [ ]:
# Technologie zu Farbe
def get_tech(name):
    if name.startswith("Methanol"):
        return "Methanol"
    elif name.startswith("Ammoniak"):
        return "Ammoniak"
    elif name.startswith("FischerTropsch"):
        return "FischerTropsch"
    else:
        return "Sonstige"

color_map = {
    "Methanol": col_ch3oh,
    "Ammoniak": col_nh3,
    "FischerTropsch": col_saf,
    "Sonstige": "gray"
}

# Modelle mit calc_cost_el sammeln
modelle_unsortiert = {
    name: globals()[name]
    for name in globals()
    if isinstance(globals()[name], pd.Series)
    and "calc_cost_el" in globals()[name]
}

# Sortierung nach calc_cost_el (absteigend)
modelle_sortiert = sorted(
    modelle_unsortiert.items(),
    key=lambda item: -item[1]["calc_cost_el"]
)

# Extraktion
categories = [name for name, _ in modelle_sortiert]
values = [model["calc_cost_el"] for _, model in modelle_sortiert]
colors = [color_map[get_tech(name)] for name in categories]

# Plot
fig, ax = plt.subplots(figsize=(12, 10))


bars = ax.barh(categories, values, color=colors, edgecolor='black')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f"{width:,.2f}",
            va='center', ha='left', fontsize=8, color='black')

# Achsen-Stil
ax.spines['top'].set_color('lightgray')
ax.spines['top'].set_linewidth(0.75)
ax.spines['top'].set_linestyle("--")

ax.spines['bottom'].set_color('black')
ax.spines['bottom'].set_linewidth(1)

ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth(1)

ax.spines['right'].set_color('lightgray')
ax.spines['right'].set_linewidth(0.75)
ax.spines['right'].set_linestyle("--")

ax.set_axisbelow(True)
ax.grid(which='major', linestyle='--', linewidth=0.75, color='lightgray')
ax.grid(which='minor', linestyle=':', linewidth=0.25, color='lightgray')

plt.xlabel("Levelized cost in ct/kWh", fontsize=10)
plt.title("Levelized Cost of Electricity", fontsize=10)
plt.xticks(fontsize=9)
plt.yticks(fontsize=7)
ax.set_xlim([0, 7])
ax.set_ylim([-0.5, 36.5])

plt.tight_layout()
plt.savefig("Vergleich_Bilder/calc_cost_el_bar.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
color_map = {
    "Methanol": col_el,
    "Ammoniak": col_el,
    "FischerTropsch": col_el
}

# === Modelle mit calc_cost_el sammeln ===
modelle_unsortiert = {
    name: globals()[name]
    for name in globals()
    if isinstance(globals()[name], pd.Series)
    and "calc_cost_el" in globals()[name]
}

# === Gruppieren nach Technologie ===
def get_tech(name):
    if name.startswith("Methanol"):
        return "Methanol"
    elif name.startswith("Ammoniak"):
        return "Ammoniak"
    elif name.startswith("FischerTropsch"):
        return "FischerTropsch"
    return "Sonstige"

modelle_gruppiert = {
    "Ammoniak": {},
    "Methanol": {},
    "FischerTropsch": {}
}

for name, df in modelle_unsortiert.items():
    tech = get_tech(name)
    if tech in modelle_gruppiert:
        modelle_gruppiert[tech][name] = df

# === Plots pro Technologie ===
for tech, gruppe in modelle_gruppiert.items():
    if not gruppe:
        continue  # überspringen wenn leer

    # Sortieren nach cost_el absteigend
    gruppe_sortiert = sorted(gruppe.items(), key=lambda item: -item[1]["calc_cost_el"])

    # Extrahiere Namen, Werte, Farben
    categories = [name for name, _ in gruppe_sortiert]
    values = [df["calc_cost_el"] for _, df in gruppe_sortiert]
    colors = [color_map[tech]] * len(values)

    # Plot
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.barh(categories, values, color=colors, edgecolor='black')

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.1,
                bar.get_y() + bar.get_height() / 2,
                f"{width:,.2f}",
                va='center', ha='left', fontsize=8, color='black')

    ax.set_xlabel("Levelized cost in ct/kWh", fontsize=10)
    ax.set_title(f"Levelized Cost of Electricity – {tech}", fontsize=11)
    ax.set_xlim(0, 7)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linestyle='--', color='lightgray', linewidth=0.75)
    plt.tight_layout()

    # Speichern
    #plt.savefig(f"Bilder/calc_cost_el_{tech.lower()}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
color_map = {
    "Methanol": col_el,
    "Ammoniak": col_el,
    "FischerTropsch": col_el
}

# Gruppierung vorbereiten (aus vorherigem Schritt)
modelle_gruppiert = {
    "Ammoniak": {},
    "Methanol": {},
    "FischerTropsch": {}
}

# Modelle mit calc_cost_el aus globalem Namensraum holen
for name, df in globals().items():
    if isinstance(df, pd.Series) and "calc_cost_el" in df.index:
        if name.startswith("Methanol"):
            modelle_gruppiert["Methanol"][name] = df
        elif name.startswith("Ammoniak"):
            modelle_gruppiert["Ammoniak"][name] = df
        elif name.startswith("FischerTropsch"):
            modelle_gruppiert["FischerTropsch"][name] = df

# Setup für Subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 0.5 * max(len(g) for g in modelle_gruppiert.values())))
techs = ["Ammoniak", "Methanol", "FischerTropsch"]

for ax, tech in zip(axes, techs):
    gruppe = modelle_gruppiert[tech]
    if not gruppe:
        continue

    # Sortieren nach calc_cost_el (absteigend)
    gruppe_sortiert = sorted(gruppe.items(), key=lambda item: -item[1]["calc_cost_el"])
    categories = [name for name, _ in gruppe_sortiert]
    values = [df["calc_cost_el"] for _, df in gruppe_sortiert]
    colors = [color_map[tech]] * len(values)

    # Plot
    bars = ax.barh(categories, values, color=colors, edgecolor='black')
    ax.set_title(tech)
    ax.set_xlim(0, 8)
    ax.invert_yaxis()
    ax.grid(axis='x', linestyle='--', color='lightgray', linewidth=0.75)
    ax.set_xlabel("ct/kWh")

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.1,
                bar.get_y() + bar.get_height() / 2,
                f"{width:.2f}",
                va='center', ha='left', fontsize=8)

# Titel und Layout
plt.suptitle("Levelized Cost of Electricity – je Technologie", fontsize=13)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
#lt.savefig("Bilder/calc_cost_el_subplots.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_el",
    farbe=col_el,
    xlim=(0, 8),
    ylim=(-0.5, 36.5),
    save_path="Vergleich_Bilder/calc_cost_el_bar.png",
    xlabel="Levelized cost in ct/kWh",
    title="Levelized Cost of Electricity",
    multiplikator=1
)


## 4.2 Levelized Cost of Heat

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_q",
    farbe=col_q,
    xlim=(0, 4),
    ylim=(-0.5, 24.5),
    save_path="Vergleich_Bilder/calc_cost_q_bar.png",
    xlabel="Levelized cost in ct/kWh_th",
    title="Levelized Cost of Heat",
    multiplikator=100
)

## 4.3 Levelized cost of Hydrogen

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_h2",
    farbe=col_h2,
    xlim=(0, 5),
    ylim=(-0.5, 36.5),
    save_path="Vergleich_Bilder/calc_cost_h2_bar.png",
    xlabel="Levelized cost in €/kg_H2",
    title="Levelized Cost of Hydrogen",
    multiplikator=1
)

## 4.4 Levelized cost of CO2

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_co2",
    farbe=col_co2,
    xlim=(0, 1),
    ylim=(-0.5, 24.5),
    save_path="Vergleich_Bilder/calc_cost_co2_bar.png",
    xlabel="Levelized cost in €/kg_CO2",
    title="Levelized Cost of Carbon Dioxide",
    multiplikator=1
)

## 4.5 Levelized cost of Methanol, Ammonia and SAF

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_ch3oh",
    farbe=col_ch3oh,
    xlim=(0, 2),
    ylim=(-0.5, 13.5),
    save_path="Vergleich_Bilder/calc_cost_ch3oh_bar.png",
    xlabel="Levelized cost in €/kg_CH3OH",
    title="Levelized Cost of Methanol",
    multiplikator=1/1000
)

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_saf",
    farbe=col_saf,
    xlim=(0, 8),
    ylim=(-0.5, 10.5),
    save_path="Vergleich_Bilder/calc_cost_saf_bar.png",
    xlabel="Levelized cost in €/kg_SAF",
    title="Levelized Cost of SAF",
    multiplikator=1/1000
)

In [ ]:
plot_modelvergleich(
    parameter="calc_cost_nh3",
    farbe=col_nh3,
    xlim=(0, 2),
    ylim=(-0.5, 11.5),
    save_path="Vergleich_Bilder/calc_cost_nh3_bar.png",
    xlabel="Levelized cost in €/kg_NH3",
    title="Levelized Cost of Ammonia",
    multiplikator=1/1000
)

# 5 Verkaufspreise

## 5.1 Maximaler Verkaufspreis (ohne Vorverkauf)

In [ ]:
plot_modelvergleich(
    parameter="calc_maxprice",
    farbe=col_sell,
    xlim=(0, 14),
    ylim=(-0.5, 36.5),
    save_path="Vergleich_Bilder/calc_maxprice_bar.png",
    xlabel="Maximum Price in €/kg",
    title="Maximum Price of SAF, Methanol or Ammonia",
    multiplikator=1
)

## 5.2 Minimaler Verkaufspreis (mit Vorverkauf)

In [ ]:
plot_modelvergleich(
    parameter="calc_price",
    farbe=col_sell,
    xlim=(0, 14),
    ylim=(-0.5, 36.5),
    save_path="Vergleich_Bilder/calc_price_bar.png",
    xlabel="Minimum Price in €/kg",
    title="Minimum Price of SAF, Methanol or Ammonia",
    multiplikator=1
)